In [11]:
import pandas as pd
import torch
from transformers import AutoTokenizer, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
import os
from torch import Tensor
from torch.utils.data import DataLoader
import faiss
import json
from beir.datasets.data_loader import GenericDataLoader

In [17]:
def stream_msmarco_chunks(path, chunk_size=5000):
    buffer = []
    with open(path, "r") as f:
        for line in f:
            doc = json.loads(line)
            buffer.append((doc["_id"], doc["text"]))

            if len(buffer) == chunk_size:
                yield buffer
                buffer = []

        if buffer:
            yield buffer

# # Loading Data
data_path = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco/corpus.jsonl"

In [13]:
# Selecting Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Loading Passage Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
passage_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/passage_encoder"
).to(device)
passage_encoder.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [14]:
# # Custom MSMARCO class
# class MSMARCO:
#     def __init__(self, passages):
    
#         '''
#         DataLoader for Vector Database
#         '''

#         self.passages = passages
    
#     def __len__(self):
#         return len(self.passages)
    
#     def __getitem__(self, idx):
#         passage = self.passages[idx]
#         return {"passage": passage}

In [15]:
# dataset = MSMARCO(passages)

In [18]:
# def collate_fn(batch:int, tokenizer:object, max_length:int=128) -> dict:
# 
#     passages = [x['passage'] for x in batch]
#     p_tok = tokenizer(passages, 
#                       padding=True,
#                       truncation=True, 
#                       max_length=max_length,
#                       return_tensors='pt'
#                       )
#     return {'passages': p_tok}

In [ ]:
# dataloader = DataLoader(dataset,
#                         batch_size=32,
#                         shuffle=False,
#                         num_workers=4,
#                         pin_memory=True)

In [ ]:
DIM = 768
NLIST = 4096
M = 64
NBITS = 8

TRAIN_SIZE = 100_000 
BATCH_SIZE = 64
CHUNK_SIZE = 5000

quantizer = faiss.IndexFlatIP(DIM)
index = faiss.IndexIVFPQ(
    quantizer,
    DIM,
    NLIST,
    M,
    NBITS
)

train_buf = []
train_count = 0
train_ids = []

meta_file = open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl", "w")
global_idx = 0

for chunk in stream_msmarco_chunks(data_path, 5000):

    for i in range(0, len(chunk), BATCH_SIZE):
        batch = chunk[i : i + BATCH_SIZE]
        passages = [x[1] for x in batch]
        doc_ids = [x[0] for x in batch]

    
        with torch.no_grad():
            inputs = tokenizer(
                passages,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors='pt'
            ).to(device)

            emb = passage_encoder(**inputs).last_hidden_state[:, 0]

        emb = emb.float().cpu().numpy()
        emb = np.ascontiguousarray(emb, dtype=np.float32)
        faiss.normalize_L2(emb)

        if not index.is_trained:
            train_buf.append(emb)
            train_ids.extend(doc_ids)
            train_count += emb.shape[0]

            if train_count >= TRAIN_SIZE:
                print(f"Training FAISS index on {train_count} vectors")

                train_vecs = np.vstack(train_buf)
                index.train(train_vecs)
                index.add(train_vecs)

                # WRITE METADATA FOR TRAINING VECTORS
                for doc_id in train_ids:
                    meta_file.write(json.dumps({
                        "idx": global_idx,
                        "doc_id": doc_id
                    }) + "\n")
                    global_idx += 1

                del train_vecs
                del train_buf
                del train_ids
                train_buf = None
                train_ids = None

                print("FAISS index trained")

            continue

        if index.is_trained:
            index.add(emb)

        for doc_id in doc_ids:
            meta_file.write(json.dumps({
                "idx": global_idx,
                "doc_id": doc_id
            }) + "\n")
            global_idx += 1

        del emb, inputs
        
    torch.cuda.empty_cache()
    del chunk


meta_file.close()
faiss.write_index(index, "passage.index")

### Sanity Check

In [22]:
meta_file = open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl", "w")

In [23]:
sum(1 for _ in open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl"))

0